In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs into DataFrames and parse timestamps as UTC datetimes (Unix seconds)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Ensure 'value' columns are numeric where applicable
for df in (metric_df, trace_df, log_df):
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')

# 1) metric.csv: group by cmdb_id and kpi_name -> count and global P95 of 'value'
g_metric = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
metric_summary = g_metric.agg(count='size', p95_value=lambda x: np.nan if x.dropna().empty else x.quantile(0.95)).reset_index()
metric_summary = metric_summary.sort_values('count', ascending=False).reset_index(drop=True)
df_metric_top50 = metric_summary.head(50)

# 2) trace.csv: group by cmdb_id and trace_name -> count and global P95 of 'value'
g_trace = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
trace_summary = g_trace.agg(count='size', p95_value=lambda x: np.nan if x.dropna().empty else x.quantile(0.95)).reset_index()
trace_summary = trace_summary.sort_values('count', ascending=False).reset_index(drop=True)
df_trace_top50 = trace_summary.head(50)

# 3) log.csv: group by cmdb_id and log_name -> count and global P95 of 'value'
g_log = log_df.groupby(['cmdb_id', 'log_name'])['value']
log_summary = g_log.agg(count='size', p95_value=lambda x: np.nan if x.dropna().empty else x.quantile(0.95)).reset_index()
log_summary = log_summary.sort_values('count', ascending=False).reset_index(drop=True)
df_log_top50 = log_summary.head(50)

# 4) error_logs.csv: counts per cmdb_id, earliest and latest timestamp (ISO UTC strings)
if not error_logs_df.empty:
    err_grp = error_logs_df.groupby('cmdb_id')['timestamp']
    error_summary = err_grp.agg(count='size', earliest_ts='min', latest_ts='max').reset_index()
    # Convert timestamps to ISO UTC strings with trailing 'Z'
    error_summary['earliest_ts'] = error_summary['earliest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    error_summary['latest_ts'] = error_summary['latest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
else:
    error_summary = pd.DataFrame(columns=['cmdb_id', 'count', 'earliest_ts', 'latest_ts'])

df_error_summary = error_summary.sort_values('count', ascending=False).head(50).reset_index(drop=True)

# Display compact results (each variable is a DataFrame limited to at most 50 rows)
df_metric_top50, df_trace_top50, df_log_top50, df_error_summary

```
Out[1]:
```
summary = (
    "Summary of telemetry (no incident-time filtering):\n"
    "- metric.csv: 50 top groups returned. The 'carts' service appears frequently among the highest-count metric groups (top KPIs: cpu, latency-50, latency-90, mem, socket). "
    "Notable P95 values include carts.mem (~2.15e8), carts.socket (~11.74), and several services with P95 CPU ~1.0–1.7. Many metric groups have 25 samples each.\n"
    "- trace.csv: no trace groups found (empty result).\n"
    "- log.csv: 22 log groups returned (top records include catalogue, orders, front-end). log.row_count P95 is high for front-end and orders; log.error_count P95 is 0 for listed services.\n"
    "- error_logs.csv: no error log records found (empty result).\n\n"
    "In short: metrics are populated (with 'carts' standing out among top groups and high P95 memory/socket values), traces and error logs are empty, and logs show high row counts but no recorded error_count in the top groups."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count     p95_value
0          carts         cpu     25  1.697045e+00
1          carts  latency-50     25  1.989833e-02
2          carts  latency-90     25  4.307411e-02
3          carts         mem     25  2.149138e+08
4          carts      socket     25  1.174333e+01
..           ...         ...    ...           ...
45  queue-master         cpu     25  1.439154e+00
46  queue-master      diskio     25  5.484102e+04
47  queue-master         mem     25  2.569697e+08
48  queue-master      socket     25  3.000000e+00
49      rabbitmq         cpu     25  9.155857e-01

[50 rows x 4 columns], Empty DataFrame
Columns: [cmdb_id, trace_name, count, p95_value]
Index: [],        cmdb_id         log_name  count  p95_value
0    catalogue  log.error_count     25        0.0
1    catalogue    log.row_count     25      169.6
2       orders    log.row_count     25      582.0
3       orders  log.error_count     25        0.0
4    front-end    log.row_count     25     2057.6
..         ...              ...    ...        ...
17       carts    log.row_count     19      127.5
18  session-db  log.error_count      5        0.0
19  session-db    log.row_count      5        5.0
20     user-db  log.error_count      1        0.0
21     user-db    log.row_count      1        1.0

[22 rows x 4 columns], Empty DataFrame
Columns: [cmdb_id, count, earliest_ts, latest_ts]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse previously loaded DataFrames: metric_df, log_df
# Ensure timestamp parsed and in UTC
for df in (metric_df, log_df):
    if 'timestamp' in df.columns and not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Candidate components
candidates = ['carts', 'catalogue', 'orders', 'payment', 'user']

# Incident window (UTC)
start = pd.Timestamp('2024-01-20 10:47:52', tz='UTC')
end   = pd.Timestamp('2024-01-20 11:17:52', tz='UTC')

# ---------- METRIC ANOMALY DETECTION ----------
# 1) Compute global P95 per (cmdb_id, kpi_name) for candidate components using full series
metric_thresh = (
    metric_df[metric_df['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter metric rows to incident window for candidate components
metric_window = metric_df[
    (metric_df['cmdb_id'].isin(candidates)) &
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end)
].copy()

# Merge thresholds into windowed rows
metric_window = metric_window.merge(metric_thresh, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Detect anomalies where value >= group P95 (use >= as requested)
metric_anoms = metric_window[metric_window['value'] >= metric_window['p95_value']].copy()

if not metric_anoms.empty:
    metric_agg = (
        metric_anoms
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            anomaly_count=('value', 'size'),
            earliest_ts=('timestamp', 'min'),
            latest_ts=('timestamp', 'max'),
            max_value_in_window=('value', 'max'),
            p95_value=('p95_value', 'first')
        )
    )
    # Convert timestamps to ISO UTC strings
    metric_agg['earliest_ts'] = metric_agg['earliest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    metric_agg['latest_ts'] = metric_agg['latest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    metric_result = metric_agg.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
else:
    # explicit empty result set with required columns
    metric_result = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_ts','latest_ts','max_value_in_window','p95_value'])

# ---------- LOG ANOMALY DETECTION ----------
# 1) Compute global P95 and P15 per (cmdb_id, log_name) for candidate components using full series
log_thresh = (
    log_df[log_df['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(
        p95_value=lambda x: x.quantile(0.95),
        p15_value=lambda x: x.quantile(0.15)
    )
    .reset_index()
)

# 2) Filter log rows to incident window for candidate components
log_window = log_df[
    (log_df['cmdb_id'].isin(candidates)) &
    (log_df['timestamp'] >= start) &
    (log_df['timestamp'] <= end)
].copy()

# Merge thresholds into windowed rows
log_window = log_window.merge(log_thresh, on=['cmdb_id', 'log_name'], how='left')

# 3) Detect anomalies:
# - low_row_count: log_name == 'log.row_count' AND value <= P15
# - high_error: log_name == 'log.error_count' AND value >= P95
low_mask = (log_window['log_name'] == 'log.row_count') & (log_window['value'] <= log_window['p15_value'])
high_mask = (log_window['log_name'] == 'log.error_count') & (log_window['value'] >= log_window['p95_value'])

low_anoms = log_window[low_mask].copy()
high_anoms = log_window[high_mask].copy()

# Aggregate low_row_count anomalies
if not low_anoms.empty:
    low_agg = (
        low_anoms
        .groupby(['cmdb_id', 'log_name'], as_index=False)
        .agg(
            anomaly_count=('value', 'size'),
            earliest_ts=('timestamp', 'min'),
            latest_ts=('timestamp', 'max'),
            extremal_value_in_window=('value', 'min'),
            p95_value=('p95_value', 'first'),
            p15_value=('p15_value', 'first')
        )
    )
    low_agg['anomaly_type'] = 'low_row_count'
else:
    low_agg = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count','earliest_ts','latest_ts','extremal_value_in_window','p95_value','p15_value','anomaly_type'])

# Aggregate high_error anomalies
if not high_anoms.empty:
    high_agg = (
        high_anoms
        .groupby(['cmdb_id', 'log_name'], as_index=False)
        .agg(
            anomaly_count=('value', 'size'),
            earliest_ts=('timestamp', 'min'),
            latest_ts=('timestamp', 'max'),
            extremal_value_in_window=('value', 'max'),
            p95_value=('p95_value', 'first'),
            p15_value=('p15_value', 'first')
        )
    )
    high_agg['anomaly_type'] = 'high_error'
else:
    high_agg = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count','earliest_ts','latest_ts','extremal_value_in_window','p95_value','p15_value','anomaly_type'])

# Combine low and high aggregates
log_result_all = pd.concat([low_agg, high_agg], ignore_index=True, sort=False)

if not log_result_all.empty:
    # Convert timestamps to ISO UTC strings
    log_result_all['earliest_ts'] = log_result_all['earliest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    log_result_all['latest_ts'] = log_result_all['latest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    log_result = log_result_all.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
else:
    # explicit empty result set with required columns
    log_result = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_type','anomaly_count','earliest_ts','latest_ts','extremal_value_in_window','p95_value','p15_value'])

# Display compact results (up to 20 rows each)
metric_result.head(20), log_result.head(20)

```
Out[2]:
```
summary = (
    "Summary of anomalies in the incident window (2024-01-20 10:47:52–11:17:52 UTC):\n\n"
    "Metrics (carts, catalogue, orders, payment, user):\n"
    "- catalogue.socket and payment.socket are the strongest signals: each had 25 anomaly samples spanning ~10:50–11:14 with max values equal to their global P95 (catalogue.socket max=6, payment.socket max=5).\n"
    "- user.socket had 9 spikes between 10:50–10:58 (max=19).\n"
    "- carts shows several small-number anomalies (cpu, socket, workload, latency, mem) — each with 1–2 samples; carts.cpu exceeded its global P95 at ~10:58–10:59.\n"
    "- orders shows multiple sporadic anomalies (cpu, diskio, latency, error, mem), generally 2 samples each.\n\n"
    "Logs:\n"
    "- log.error_count: global P95 is 0 for many services, so the detection flagged many rows as 'high_error' (payment, user, orders, catalogue: 25 samples; carts: 19). However the actual extremal error values in-window are 0 — this is a threshold artifact (P95==0), not evidence of elevated error counts.\n"
    "- log.row_count: low_row_count events were detected for carts (5 samples, min 25), catalogue (4 samples, min 23), user (4 samples, min 73), orders (4 samples, min 14), payment (4 samples, min 13) indicating brief drops in traffic/log-row volume.\n\n"
    "Overall interpretation and next steps:\n"
    "- The most actionable signals point to socket-related anomalies on catalogue and payment (also notable on user). Investigate networking or service socket issues for those services.\n"
    "- Also investigate brief drops in request/row counts for carts, catalogue, user, orders, and payment (could indicate partial outages, routing/traffic issues or downstream backpressure).\n"
    "- Do not treat the log.error_count 'high_error' flags as real errors without verifying values, because the global P95 was 0 and the in-window error values are 0.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name  anomaly_count           earliest_ts             latest_ts  max_value_in_window     p95_value
0   catalogue      socket             25  2024-01-20T10:50:00Z  2024-01-20T11:14:00Z         6.000000e+00  6.000000e+00
1     payment      socket             25  2024-01-20T10:50:00Z  2024-01-20T11:14:00Z         5.000000e+00  5.000000e+00
2        user      socket              9  2024-01-20T10:50:00Z  2024-01-20T10:58:00Z         1.900000e+01  1.900000e+01
3       carts         cpu              2  2024-01-20T10:58:00Z  2024-01-20T10:59:00Z         4.383984e+00  1.697045e+00
4       carts      socket              2  2024-01-20T11:00:00Z  2024-01-20T11:04:00Z         1.185000e+01  1.174333e+01
5       carts    workload              2  2024-01-20T10:53:00Z  2024-01-20T10:54:00Z         8.808933e+00  8.477733e+00
6       carts  latency-90              2  2024-01-20T11:07:00Z  2024-01-20T11:09:00Z         4.318496e-02  4.307411e-02
7       carts  latency-50              2  2024-01-20T11:05:00Z  2024-01-20T11:07:00Z         1.992116e-02  1.989833e-02
8   catalogue  latency-50              2  2024-01-20T11:02:00Z  2024-01-20T11:03:00Z         3.115631e-03  3.078654e-03
9   catalogue  latency-90              2  2024-01-20T11:02:00Z  2024-01-20T11:03:00Z         4.808135e-03  4.741577e-03
10  catalogue         mem              2  2024-01-20T10:59:00Z  2024-01-20T11:00:00Z         6.310707e+06  6.306051e+06
11  catalogue    workload              2  2024-01-20T10:53:00Z  2024-01-20T11:14:00Z         4.344367e+00  4.312640e+00
12     orders         cpu              2  2024-01-20T10:50:00Z  2024-01-20T11:02:00Z         5.958260e+00  3.708939e+00
13     orders      diskio              2  2024-01-20T10:50:00Z  2024-01-20T10:51:00Z         1.962641e+05  7.785663e+04
14  catalogue         cpu              2  2024-01-20T11:01:00Z  2024-01-20T11:02:00Z         1.921904e-01  1.864738e-01
15      carts         mem              2  2024-01-20T11:12:00Z  2024-01-20T11:14:00Z         2.149262e+08  2.149138e+08
16     orders  latency-50              2  2024-01-20T11:08:00Z  2024-01-20T11:11:00Z         7.308275e-01  7.107427e-01
17     orders       error              2  2024-01-20T11:03:00Z  2024-01-20T11:10:00Z         3.332500e-01  1.595811e-01
18     orders  latency-90              2  2024-01-20T11:05:00Z  2024-01-20T11:10:00Z         3.662431e+00  3.362813e+00
19     orders         mem              2  2024-01-20T11:02:00Z  2024-01-20T11:03:00Z         3.298680e+08  3.287328e+08,      cmdb_id         log_name  anomaly_count           earliest_ts             latest_ts  extremal_value_in_window  p95_value  p15_value   anomaly_type
0    payment  log.error_count             25  2024-01-20T10:50:00Z  2024-01-20T11:14:00Z                         0        0.0        0.0     high_error
1       user  log.error_count             25  2024-01-20T10:50:00Z  2024-01-20T11:14:00Z                         0        0.0        0.0     high_error
2     orders  log.error_count             25  2024-01-20T10:50:00Z  2024-01-20T11:14:00Z                         0        0.0        0.0     high_error
3  catalogue  log.error_count             25  2024-01-20T10:50:00Z  2024-01-20T11:14:00Z                         0        0.0        0.0     high_error
4      carts  log.error_count             19  2024-01-20T10:52:00Z  2024-01-20T11:14:00Z                         0        0.0        0.0     high_error
5      carts    log.row_count              5  2024-01-20T10:57:00Z  2024-01-20T11:08:00Z                        25      127.5       25.0  low_row_count
6  catalogue    log.row_count              4  2024-01-20T10:50:00Z  2024-01-20T11:14:00Z                        23      169.6      152.6  low_row_count
7       user    log.row_count              4  2024-01-20T10:50:00Z  2024-01-20T11:14:00Z                        73      621.2      553.2  low_row_count
8     orders    log.row_count              4  2024-01-20T10:50:00Z  2024-01-20T11:05:00Z                        14      582.0      113.2  low_row_count
9    payment    log.row_count              4  2024-01-20T10:50:00Z  2024-01-20T11:14:00Z                        13      106.8       94.6  low_row_count)```
```